# 01 — Exploration des données (EDA)

**Projet** : Détection de fraude sur transactions de paiement avec scikit-learn
**Cas métier** : Détecter **sans étiquette fiable et à jour** les transactions anormales — la fraude confirmée n'est connue qu'après 30 à 90 jours (chargeback), donc un modèle supervisé apprend toujours sur une vérité partielle et biaisée. Objectif : produire un score de risque continu, ordonner les transactions et transmettre aux analystes un volume d'alertes compatible avec leur capacité, en maximisant la fraude capturée.
**Jeu de données** : `payment_transactions` — Transactions de paiement et fraude observée

> Flux de transactions d'un PSP français sur 90 jours, avec le contexte marchand, porteur et technique de chaque paiement. Les données sont générées par un modèle latent : chaque transaction légitime suit des distributions réalistes par catégorie et par canal, tandis que quatre **modes opératoires** de fraude (card-not-present, prise de compte, identité synthétique, fraude amicale) déforment des sous-ensembles précis de variables (vélocité, montant, empreinte d'appareil, authentification, géographie). Du bruit irréductible, des valeurs manquantes et des outliers **légitimes** (achats de luxe, paiements professionnels) sont injectés : un détecteur qui ne fait que repérer les gros montants se fait piéger. Deux colonnes de métadonnées (`is_fraud`, `fraud_scheme`) permettent de mesurer la performance sans jamais entrer dans les features.

## Objectifs pédagogiques

1. Charger un jeu de données tabulaire et en établir le profil (types, manquants, doublons).
1. Lire une distribution : détecter déséquilibre, outliers et colinéarité **avant** de modéliser.
1. Relier chaque observation statistique à une conséquence métier ou de modélisation.
1. Produire les figures qui serviront de référence dans les notebooks suivants.

**Objectifs transverses du dépôt**

- Comprendre pourquoi la fraude se détecte sans supervision : étiquette tardive (chargeback à J+30), partielle et biaisée par les règles existantes.
- Construire un pipeline sans fuite : `is_fraud` et `fraud_scheme` sont des métadonnées exclues des features par configuration.
- Lire les bonnes métriques en forte imbalance : PR AUC et lift plutôt qu'accuracy et ROC AUC, et toujours relativement au plancher (= prévalence).

## 0. Environnement

Toute la configuration vient de **Hydra** (`conf/`) : aucune valeur métier n'est codée en dur
dans ce notebook. Si `data/raw` est vide, le générateur synthétique du projet prend le relais
(voir `make data`).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet configure loguru au premier `get_logger()` appelé par `src`. On prend la main ici,
# au niveau WARNING : sans cela, chaque cellule d'entraînement noierait ses tableaux sous les
# lignes INFO de production. Les avertissements réels restent visibles — c'est l'essentiel.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (12000 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 12000

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=3",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 0.42)")
print(f"Cible             : {CONFIG.data.target or 'aucune (apprentissage non supervisé)'}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

**Pourquoi ce bloc d'initialisation**

- `CONFIG` est l'objet **Pydantic** validé : une clé incohérente échoue ici, pas en production.
- Les notebooks travaillent sur un échantillon réduit pour rester rapides ; `make train` utilise `data.n_samples` complet.
- `NB_PATHS` isole les écritures du notebook dans `outputs/notebooks`.

## 1. Chargement et premier contact

On ne regarde jamais un dataset sans vérifier trois choses : sa **forme** (lignes x colonnes),
ses **types** (un numérique lu comme texte casse tout) et ses **premières lignes** (les valeurs
ont-elles du sens métier ?).

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

**Ce qu'il faut retenir**

- Le contrat `payment_transactions` est documenté dans `data/README.md` : chaque colonne y a une signification métier.
- Les identifiants et horodatages ne sont **pas** des features : ils servent à tracer et à splitter.

In [ ]:
from src.data.schemas import describe_schema, validation_report

# Types déclarés (contrat Pandera) vs types réellement lus : toute divergence est un signal.
contract = describe_schema("raw")
observed = pd.DataFrame({"dtype_lu": {str(k): str(v) for k, v in raw.dtypes.items()}})
contract.join(observed)[["dtype", "dtype_lu", "nullable", "unique", "checks"]]

**Ce qu'il faut retenir**

- La colonne `dtype` vient du **contrat**, `dtype_lu` de la source : elles doivent correspondre.
- Les `checks` (bornes, valeurs autorisées) sont la mémoire des règles métier — ils seront testés au notebook 02.

In [ ]:
report = validation_report(raw)
summary = pd.DataFrame(
    {
        "indicateur": [
            "lignes",
            "colonnes",
            "cellules manquantes",
            "taux de manquants",
            "mémoire (Ko)",
        ],
        "valeur": [
            report["n_rows"],
            report["n_columns"],
            report["missing_cells"],
            f"{report['missing_rate']:.2%}",
            round(report["memory_kb"], 1),
        ],
    }
)
summary

**Ce qu'il faut retenir**

- Un taux de manquants global faible peut cacher une colonne très incomplète : regarder **par colonne**.
- La mémoire indique si le dataset tient en RAM (sinon : pyarrow, chunking ou échantillonnage).

## 2. Valeurs manquantes

Où, combien, et surtout : **manquant au hasard ou pas** ? Un manquant informatif (ex. score de satisfaction non renseigné par les clients mécontents) est un signal, pas seulement un problème technique.

In [ ]:
missing = raw.isna().sum()
missing_frame = (
    pd.DataFrame({"manquants": missing, "taux": (missing / len(raw)).round(4)})
    .loc[lambda frame: frame["manquants"] > 0]
    .sort_values("manquants", ascending=False)
)
missing_frame

In [ ]:
if missing_frame.empty:
    print("Aucune valeur manquante dans cet échantillon.")
else:
    fig, axis = plt.subplots(figsize=(7.5, 0.55 * len(missing_frame) + 1.6))
    axis.barh(missing_frame.index[::-1], missing_frame["taux"][::-1] * 100, color="#d1495b")
    axis.set_xlabel("Cellules manquantes (%)")
    axis.set_title("Valeurs manquantes par colonne")
    fig.tight_layout()
    plt.show()

**Ce qu'il faut retenir**

- L'imputation doit être **apprise sur le train** (moyenne/médiane/constante) puis appliquée aux autres splits.
- Ajouter un indicateur binaire « valeur manquante » est souvent rentable quand le manquant est informatif.
- Notes du générateur : Valeurs manquantes volontaires sur `device_age_days` (~4 %), `session_duration_sec` (~6 %) et `billing_shipping_distance_km` (~5 %).; Outliers légitimes (~2,5 % des lignes) : achats de luxe, paiements professionnels de fin de mois, voyageurs fréquents..

## 3. Distributions numériques

In [ ]:
numeric_columns = [column for column in raw.columns if pd.api.types.is_numeric_dtype(raw[column])]
numeric_columns = [column for column in numeric_columns if column != CONFIG.data.target]

n_plots = len(numeric_columns)
n_cols = 3
n_rows = int(np.ceil(n_plots / n_cols)) if n_plots else 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.0 * n_cols, 2.7 * n_rows))
for axis, column in zip(np.atleast_1d(axes).ravel(), numeric_columns, strict=False):
    raw[column].hist(bins=30, ax=axis, color="#005f73", edgecolor="white")
    axis.set_title(column, fontsize=9)
    axis.tick_params(labelsize=7)
for axis in np.atleast_1d(axes).ravel()[len(numeric_columns) :]:
    axis.axis("off")
fig.suptitle("Distributions des variables numériques", y=1.005)
fig.tight_layout()
plt.show()

In [ ]:
raw[numeric_columns].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T.round(2)

**Ce qu'il faut retenir**

- Une distribution très asymétrique (max ≫ p99) justifie un **winsorising** ou un `log1p` plutôt qu'une suppression d'outliers.
- Des échelles hétérogènes (euros, Go, unités) imposent un **scaling** pour les modèles sensibles à la distance (SVM, k-NN, réseaux).
- Comparer `mean` et `50%` : un écart important signale une queue lourde.

## 4. Variables catégorielles

In [ ]:
categorical_columns = [
    column
    for column in raw.columns
    if not pd.api.types.is_numeric_dtype(raw[column])
    and column not in [*CONFIG.data.drop_columns, str(CONFIG.data.target)]
]

for column in categorical_columns:
    counts = raw[column].astype(str).value_counts()
    print(f"--- {column} ({len(counts)} modalités) ---")
    print((counts / len(raw)).map("{:.1%}".format).to_string())

**Ce qu'il faut retenir**

- Une modalité ultra-rare (< 1 %) doit être regroupée dans un bucket `rare` : sinon l'encodage one-hot crée des colonnes quasi vides et instables.
- Une cardinalité élevée (identifiants, codes postaux) appelle un **target encoding** régularisé plutôt qu'un one-hot.

## 5. Pas de cible à prédire : qu'est-ce qu'une anomalie ici ?

Ce projet n'a **aucune variable cible** (`target: null`). Le détecteur est entraîné sans
étiquette, et les étiquettes ne servent qu'à *mesurer* la performance. Pourquoi ?

1. **l'étiquette arrive tard** — la fraude n'est confirmée qu'au chargeback, 30 à 90 jours après
   la transaction. Un modèle supervisé apprendrait donc sur une vérité partielle et périmée ;
2. **l'étiquette est biaisée** — seules les transactions déjà alertées par les règles existantes
   sont investiguées, donc étiquetées. Un schéma inédit est invisible dans les étiquettes ;
3. **la prévalence est très faible** — à ~1,8 % de fraude, l'accuracy est une métrique
   inexploitable : « ne rien signaler » obtient 98,2 %.

Deux colonnes de **métadonnées** existent dans ce jeu synthétique : `is_fraud` (fraude confirmée)
et `fraud_scheme` (mode opératoire). Elles sont exclues des features par `drop_columns` : elles servent
à juger le détecteur, jamais à l'entraîner.

L'exploration répond donc à trois questions : quelle est la **prévalence** (et le plancher de
performance qui en découle), une variable **seule** suffit-elle à séparer la fraude, et les valeurs
**manquantes** portent-elles de l'information ?

In [ ]:
label_column = "is_fraud"
scheme_column = "fraud_scheme"
labels = raw[label_column].to_numpy().astype(int)

prevalence = float(labels.mean())
n_frauds = int(labels.sum())
print(f"transactions            : {len(raw)}")
print(f"fraudes confirmées      : {n_frauds}")
print(f"prévalence              : {prevalence:.3%}")
print(f"plancher PR AUC         : {prevalence:.4f}   <- score aléatoire")
print()
print("Ce que vaut un détecteur qui ne signale rien :")
print(f"  accuracy : {1 - prevalence:.4f}   (flatteur et inutile)")
print("  rappel   : 0.0000   (aucune fraude capturée)")
print(f"  PR AUC   : {prevalence:.4f}   (égal au plancher)")

schemes = (
    raw[scheme_column]
    .fillna("légitime")
    .value_counts()
    .rename_axis("population")
    .to_frame("effectif")
)
schemes["part du flux"] = (schemes["effectif"] / len(raw)).round(4)
schemes["part de la fraude"] = (schemes["effectif"] / max(n_frauds, 1)).round(3)
display(schemes)

comparatif = (
    raw.groupby(raw[scheme_column].fillna("légitime"), observed=True)
    .agg(
        montant_médian=("amount_eur", "median"),
        vélocité_moyenne=("transactions_24h", "mean"),
        échecs_moyens=("failed_attempts_1h", "mean"),
        part_nuit=("is_night", "mean"),
        part_3ds=("three_ds_authenticated", "mean"),
        chargebacks_moyens=("previous_chargebacks_12m", "mean"),
    )
    .round(3)
)
display(comparatif)

**Ce qu'il faut retenir**

- La prévalence fixe le **plancher** : un score aléatoire obtient une PR AUC égale à la part de fraude. Toute performance publiée doit être lue relativement à ce plancher, jamais dans l'absolu.
- L'accuracy est ici un piège : un détecteur qui ne signale rien obtient ~98 % d'accuracy et 0 % de rappel. C'est la raison du choix de la PR AUC comme métrique primaire.
- Les quatre modes opératoires n'ont pas le même profil : vélocité explosive pour le card testing, géographie incohérente pour la prise de compte, historique de chargebacks pour la fraude amicale. Aucun détecteur univarié ne peut couvrir les quatre.

In [ ]:
from sklearn.metrics import roc_auc_score

feature_columns = [column for column in raw.columns if column not in set(CONFIG.data.drop_columns)]
numeric_columns = [
    column for column in feature_columns if pd.api.types.is_numeric_dtype(raw[column])
]

rows = []
for column in numeric_columns:
    values = pd.to_numeric(raw[column], errors="coerce")
    if values.notna().sum() < 100 or values.nunique() < 2:
        continue
    filled = values.fillna(values.median()).to_numpy(dtype="float64")
    auc = float(roc_auc_score(labels, filled))
    rows.append(
        {
            "feature": column,
            "auc_univariée": max(auc, 1.0 - auc),
            "sens": "fraude = valeurs hautes" if auc >= 0.5 else "fraude = valeurs basses",
            "médiane_fraude": round(float(values[labels == 1].median()), 2),
            "médiane_légitime": round(float(values[labels == 0].median()), 2),
            "manquants_%": round(float(values.isna().mean()) * 100, 2),
        }
    )

separation = pd.DataFrame(rows).sort_values("auc_univariée", ascending=False).reset_index(drop=True)
display(separation)

fig, axis = plt.subplots(figsize=(7.8, 0.34 * len(separation) + 1.8))
axis.barh(
    separation["feature"][::-1],
    (separation["auc_univariée"][::-1] - 0.5) * 2,
    color="#0a9396",
)
axis.axvline(0.0, color="#ae2012", linestyle="--", lw=1.2, label="AUC 0,50 : aucun signal")
axis.set_xlabel("pouvoir discriminant univarié (2 x |AUC - 0,5|)")
axis.set_title("Aucune variable ne suffit : le signal est multivarié")
axis.legend(fontsize=8)
fig.tight_layout()
plt.show()

**Ce qu'il faut retenir**

- Aucune variable n'atteint une AUC univariée élevée : le signal est **multivarié** et conditionnel (un montant élevé n'est anormal que relativement au panier habituel et à la catégorie).
- Les features de vélocité (`transactions_24h`, `distinct_merchants_24h`, `failed_attempts_1h`) dominent le classement univarié : c'est cohérent avec le card testing, mais elles ne couvrent pas la fraude amicale.
- `amount_eur` seul est un mauvais détecteur : les achats de luxe légitimes occupent exactement la même zone. C'est le premier piège du générateur, et la raison pour laquelle le ratio au panier habituel existe.

In [ ]:
nullable_columns = [
    column
    for column in ("device_age_days", "session_duration_sec", "billing_shipping_distance_km")
    if column in raw.columns
]

missing = pd.DataFrame(
    {
        "manquants légitime %": [
            round(float(raw.loc[labels == 0, column].isna().mean()) * 100, 2)
            for column in nullable_columns
        ],
        "manquants fraude %": [
            round(float(raw.loc[labels == 1, column].isna().mean()) * 100, 2)
            for column in nullable_columns
        ],
    },
    index=nullable_columns,
)
missing["écart (points)"] = (missing["manquants fraude %"] - missing["manquants légitime %"]).round(
    2
)
display(missing)

# Ce que produirait une imputation silencieuse : la valeur médiane écrase le signal.
for column in nullable_columns:
    values = pd.to_numeric(raw[column], errors="coerce")
    naive = values.fillna(values.median())
    auc_naive = float(roc_auc_score(labels, naive.to_numpy(dtype="float64")))
    indicator = values.isna().to_numpy().astype(int)
    auc_indicator = float(roc_auc_score(labels, indicator)) if indicator.sum() else float("nan")
    print(
        f"{column:<32} AUC valeur imputée = {max(auc_naive, 1 - auc_naive):.3f} | "
        f"AUC indicateur de manquant = {max(auc_indicator, 1 - auc_indicator):.3f}"
    )

**Ce qu'il faut retenir**

- Les manquants sont **informatifs** (MNAR) : une empreinte d'appareil bloquée est deux fois plus fréquente en fraude. Une imputation silencieuse à la médiane détruit ce signal.
- L'indicateur de manquant porte souvent plus d'information que la valeur imputée : c'est un feature à part entière, à déclarer explicitement dans `conf/preprocessing/default.yaml`.
- `billing_shipping_distance_km` est manquant pour les retraits en magasin : ici le manquant est légitime et non risqué. Même symptôme, causes opposées — d'où la nécessité de documenter la mécanique de collecte avant d'imputer.

## 6. Colinéarité et structure

In [ ]:
correlation = raw[numeric_columns].corr(numeric_only=True)
fig, axis = plt.subplots(figsize=(6.6, 5.4))
image = axis.imshow(correlation.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1)
axis.set_xticks(
    range(len(correlation.columns)), correlation.columns, rotation=45, ha="right", fontsize=7
)
axis.set_yticks(range(len(correlation.index)), correlation.index, fontsize=7)
for row in range(correlation.shape[0]):
    for column in range(correlation.shape[1]):
        value = correlation.iloc[row, column]
        axis.text(
            column,
            row,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=6,
            color="black" if abs(value) < 0.6 else "white",
        )
fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
axis.set_title("Corrélations de Pearson (variables numériques)")
fig.tight_layout()
plt.show()

**Ce qu'il faut retenir**

- Deux features corrélées à > 0.9 n'apportent presque rien ensemble : en garder une simplifie le modèle et son explication.
- Les arbres sont robustes à la colinéarité ; les modèles linéaires/régularisés voient leurs coefficients devenir instables.
- La corrélation ne capture pas les relations **non linéaires** : la vérifier par des graphes cible vs feature.

In [ ]:
# Outliers : comptage par la règle de l'IQR (1.5 x écart interquartile).
rows = []
for column in numeric_columns:
    series = raw[column].dropna()
    if series.empty:
        continue
    low, high = series.quantile([0.25, 0.75])
    iqr = high - low
    outliers = int(((series < low - 1.5 * iqr) | (series > high + 1.5 * iqr)).sum())
    rows.append(
        {"colonne": column, "outliers_iqr": outliers, "part": outliers / max(len(series), 1)}
    )
outlier_frame = pd.DataFrame(rows).sort_values("outliers_iqr", ascending=False)
outlier_frame.head(8).round(4)

**Ce qu'il faut retenir**

- La règle IQR **signale**, elle ne tranche pas : un outlier peut être un client légitime (grand compte, pic saisonnier).
- Le winsorising (clip aux quantiles 1-99 %) conserve les lignes et les labels, contrairement à la suppression.

In [ ]:
# Intégrité : unicité de la clé et doublons complets.
key = CONFIG.data.id_column
duplicates = int(raw.duplicated().sum())
key_duplicates = int(raw[key].duplicated().sum()) if key and key in raw.columns else 0
print(f"doublons complets            : {duplicates}")
print(f"doublons sur la clé '{key}' : {key_duplicates}")
unique_keys = raw[key].nunique() if key in raw.columns else "n/a"
print(f"identifiants uniques         : {unique_keys} / {len(raw)}")

## 7. Synthèse de l'exploration

**Lectures clés de ce jeu de données**

- La prévalence est très faible (~1,8 %) : l'accuracy est inutile (98,2 % en prédisant « légitime »), il faut piloter PR AUC, rappel et précision **à budget fixé**.
- Aucune variable ne suffit : `amount_eur` élevé est aussi le signe d'achats de luxe parfaitement légitimes (3 % du flux), d'où l'intérêt des ratios (montant / panier moyen) et de la vélocité.
- La vélocité (`transactions_24h`, `distinct_merchants_24h`, `failed_attempts_1h`) est le signal le plus discriminant du card testing : un porteur légitime ne tente pas 25 paiements en une heure.
- `three_ds_authenticated` est fortement protecteur mais **non absolu** : 19 % des fraudes passent l'authentification forte (prise de compte sur un appareil déjà connu).
- La divergence `shopper_country` / `billing_country` est un signal classique, mais 14 % des transactions légitimes sont en déplacement professionnel : la règle brute génère des faux positifs massifs.
- `device_age_days` et `session_duration_sec` comportent des manquants **informatifs** (empreinte bloquée, paiement one-click) : l'imputation doit être explicite et un indicateur de manquants est souvent plus utile que la valeur imputée.
- Les quatre modes opératoires n'utilisent pas les mêmes variables : un détecteur global doit couvrir des signatures hétérogènes, ce qu'un jeu de règles figé rate.
- L'étiquette arrive avec 30 à 90 jours de retard (chargeback) : c'est la raison structurelle de l'approche non supervisée, et non un choix par défaut.
- `billing_shipping_distance_km` a une distribution bimodale (0 km ou très loin) : la traiter comme une variable continue gaussienne masque le signal.
- Quelques transactions légitimes ressemblent exactement à de la fraude (achat de luxe nocturne à l'étranger) : le bruit irréductible fixe un plafond — un PR AUC de 1,0 signerait une fuite.
- Le budget d'investigation (2 % du flux) est la vraie contrainte métier : le seuil se choisit sur la courbe rappel/précision en fonction du volume, pas sur un F1 abstrait.
- Les features de vélocité sont calculées **avant** la transaction (fenêtre glissante strictement passée) : toute agrégation incluant la transaction courante créerait une fuite temporelle.

### Décisions de modélisation issues de l'EDA

| Observation | Décision |
| --- | --- |
| Valeurs manquantes localisées | Imputation apprise sur le train (notebook 03) |
| Échelles hétérogènes | Scaling numérique obligatoire |
| Outliers légitimes | Winsorising plutôt que suppression |
| Modalités rares | Regroupement `rare` avant encodage |
| Colinéarité | Surveiller l'importance des features (notebook 04) |

**Suite** : `02_validation.ipynb` transforme ces observations en **contrats exécutables** (Pandera).